# 🏥 Farmatodo Urban Analytics & Expansion Sandbox

Este notebook permite ejecutar el pipeline de geomarketing, análisis espacial y generación del dashboard interactivo sin depender de tu computadora local, directamente en **Google Colab** o **GitHub Codespaces**.

---
### 📁 Arquitectura de Datos y Google Drive
- Los datos espaciales y censos se leen desde la carpeta `farmatodo-sandbox` en tu Google Drive (`/content/drive/MyDrive/farmatodo-sandbox`).
- Si estás en GitHub Codespaces o local, se utiliza la carpeta interna `./data`.

### 1. Detección de Entorno y Clonado (para Google Colab)

In [ ]:
import os
import sys
from pathlib import Path

# Detección de Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Ejecutando en Google Colab.")
    # Clonar repositorio si aún no está presente en el entorno de ejecución
    if not os.path.exists('/content/farmatodo-sandbox'):
        !git clone https://github.com/rodserr/farmatodo-sandbox.git /content/farmatodo-sandbox
    %cd /content/farmatodo-sandbox
    sys.path.insert(0, '/content/farmatodo-sandbox')
else:
    print("💻 Ejecutando en entorno local o GitHub Codespaces.")
    # Agregar raíz al sys.path
    repo_root = Path.cwd().resolve()
    if repo_root.name == 'notebooks':
        %cd ..
        repo_root = Path.cwd().resolve()
    sys.path.insert(0, str(repo_root))

!pip install -q python-dotenv requests pandas

### 2. Conexión con Google Drive (para Google Colab)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_data_dir = Path('/content/drive/MyDrive/farmatodo-sandbox')
    if drive_data_dir.exists():
        print(f"✅ Carpeta de datos encontrada en Google Drive: {drive_data_dir}")
        os.environ['DATA_DIR'] = str(drive_data_dir)
    else:
        print(f"⚠️ Aviso: No se encontró {drive_data_dir}. Asegúrate de haber subido la carpeta 'farmatodo-sandbox' a tu Google Drive (Mi Unidad).")

### 3. Configuración Segura de API Keys (Sin exponer credenciales en código)

- En **Google Colab**: Usa la pestaña lateral izquierda **Secrets (🔑)**. Agrega un secreto con nombre `GOOGLE_MAPS_API_KEY` y activa el switch de acceso para este notebook.
- En **Codespaces**: Puedes definir `GOOGLE_MAPS_API_KEY` en tu archivo `.env` o en los Secrets del repositorio en GitHub.

In [ ]:
if IN_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_MAPS_API_KEY')
        if api_key:
            os.environ['GOOGLE_MAPS_API_KEY'] = api_key
            print("✅ Google Maps API Key cargada con éxito desde Colab Secrets.")
        else:
            print("ℹ️ Clave no encontrada en Secrets. Se usará dataset previo o solicita ingreso manual.")
    except Exception as e:
        print(f"ℹ️ Secrets no configurados: {e}")

import config
print(f"Directorio de datos activo: {config.DATA_DIR}")

### 4. Inspección de Datos Procesados

In [ ]:
import json
import pandas as pd

# Inspeccionar Tiendas Farmatodo
if config.STORES_JSON.exists():
    with open(config.STORES_JSON, 'r', encoding='utf-8') as f:
        stores_data = json.load(f)
    df_stores = pd.DataFrame(stores_data)
    print(f"Total Tiendas Farmatodo: {len(df_stores)}")
    display(df_stores[['name', 'city_group', 'cityName', 'schedule']].head(5))

# Inspeccionar Clusters de Oportunidad
if config.OPPORTUNITY_CLUSTERS_ENRICHED.exists():
    with open(config.OPPORTUNITY_CLUSTERS_ENRICHED, 'r', encoding='utf-8') as f:
        clusters_data = json.load(f)
    df_clusters = pd.DataFrame(clusters_data)
    print(f"\nTotal Clusters de Oportunidad: {len(df_clusters)}")
    display(df_clusters[['id', 'zone_label', 'category', 'city_name', 'state_name', 'total_pop', 'avg_dist_ftd_km', 'score']].head(10))

### 5. Compilación del Dashboard Web Autónomo

In [ ]:
from scripts import build_dashboard_bundle, build_final_html

print("Compilando Dashboard...")
build_dashboard_bundle()
build_final_html()
print(f"✅ Dashboard disponible en: {config.OUTPUT_HTML}")

### 6. Visualización y Descarga del Dashboard

In [ ]:
from IPython.display import HTML, display

# En Google Colab, podemos descargar directamente el archivo generado
if IN_COLAB:
    from google.colab import files
    print("Descargando archivo HTML a tu computadora...")
    files.download(str(config.OUTPUT_HTML))
else:
    print(f"Abre el archivo directamente en tu navegador o servidor local: {config.OUTPUT_HTML}")